In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

In [3]:
IC = 99.5
A = 168183.56
B = 2374.74

In [4]:
df = pd.read_csv(f"data/MB7U8CLLTLJB30349_soh_data.csv")
print(df.shape)

max_soc = df.soc.max()
if(max_soc < 100):
    print("SOH cannot be calculated because of battery not reaching 100% SOC")
else:
    print("SOH can be calculated")

(17818, 35)
SOH can be calculated


In [5]:
df.columns

Index(['vin', 'state', 'intime', 'odometer', 'soc', 'cyclenum', 'bt1', 'bt2',
       'bt3', 'bt4', 'bv1', 'bv2', 'bv3', 'bv4', 'bv5', 'bv6', 'bv7', 'bv8',
       'bv9', 'bv10', 'bv11', 'bv12', 'bv13', 'bv14', 'bv15', 'bv16', 'bv17',
       'bv18', 'bv19', 'bv20', 'bv21', 'bv22', 'bv23', 'bv24', 'ampherehour'],
      dtype='object')

In [6]:
## discard the cell if max cell voltage of a row is more than 3.7 
cols = ['bv1', 'bv2', 'bv3', 'bv4','bv5', 'bv6', 'bv7', 'bv8', 'bv9', 'bv10',
       'bv11', 'bv12', 'bv13', 'bv14', 'bv15', 'bv16','bv17','bv18','bv19','bv20','bv21','bv22','bv23','bv24']
cols1 = ['bt1', 'bt2', 'bt3', 'bt4', 'ampherehour']

df[cols] = np.where((df[cols] > 3.7),0,df[cols])
df[cols1] = df[cols1].astype(str)

## dropping where vol is more than 3.7
df = df.drop(columns=df.columns[(df == 0.0).any()])
filter_col = [col for col in df if col.startswith('bv')]
df['max_cell_voltage'] = df[filter_col].max(axis=1)
df['max_cell'] = df[filter_col].idxmax(axis=1)
df[cols1] = df[cols1].astype(float)
df

,vin,intime,odometer,soc,cyclenum,bt1,bt2,bt3,bt4,bv1,bv2,bv3,bv5,bv7,bv8,bv17,bv18,bv19,bv20,bv21,bv22,bv23,bv24,ampherehour,max_cell_voltage,min_cell_voltage,max_cell,min_cell
0,MB7U8CLLTLJB30349,22-02-2022 14:23,31913,15.0,1,30.9,31.9,30.9,32.1,3.538,3.461,3.330,3.449,3.343,3.399,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,3.538,3.330,bv1,bv3
1,MB7U8CLLTLJB30349,22-02-2022 14:25,31913,100.0,1,30.9,31.8,30.9,32.1,3.594,3.494,3.333,3.491,3.337,3.443,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.01,3.594,3.333,bv1,bv3
2,MB7U8CLLTLJB30349,22-02-2022 14:27,31913,100.0,1,30.9,31.9,30.9,32.1,3.573,3.480,3.332,3.469,3.346,3.411,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.01,3.573,3.332,bv1,bv3
3,MB7U8CLLTLJB30349,22-02-2022 14:27,31913,100.0,1,30.9,31.9,30.9,32.1,3.573,3.480,3.332,3.469,3.346,3.411,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.01,3.573,3.332,bv1,bv3
4,MB7U8CLLTLJB30349,22-02-2022 14:29,31913,100.0,1,30.9,31.9,30.9,32.1,3.554,3.467,3.330,3.456,3.343,3.403,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.04,3.554,3.330,bv1,bv3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17813,MB7U8CLLTLJB30349,21-02-2022 10:39,31913,99.8,114,33.0,31.8,33.7,32.1,3.406,3.330,3.329,3.332,3.335,3.338,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.26,3.406,3.329,bv1,bv3
17814,MB7U8CLLTLJB30349,21-02-2022 10:41,31913,99.7,114,33.0,31.6,33.7,32.1,3.391,3.327,3.327,3.330,3.333,3.337,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.31,3.391,3.327,bv1,bv2
17815,MB7U8CLLTLJB30349,21-02-2022 10:43,31913,99.7,114,33.0,31.6,33.7,32.1,3.391,3.327,3.327,3.330,3.333,3.337,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.31,3.391,3.327,bv1,bv2
17816,MB7U8CLLTLJB30349,21-02-2022 10:45,31913,99.7,114,32.9,31.6,33.5,32.0,3.378,3.325,3.325,3.330,3.333,3.335,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.35,3.378,3.325,bv1,bv2


In [8]:
df['soc'].min()

80.6

In [9]:
df = df[df['cyclenum'] == 114]
df = df.sort_values(by=['intime'])

## Filtering rows for avg Voltage greater than 3.36
filter_col = [col for col in df if col.startswith('bv')]
df['avg_V'] = df[filter_col].mean(axis=1)
df = df[df['avg_V']>3.36]
df

,vin,intime,odometer,soc,cyclenum,bt1,bt2,bt3,bt4,bv1,bv2,bv3,bv5,bv7,bv8,bv17,bv18,bv19,bv20,bv21,bv22,bv23,bv24,ampherehour,max_cell_voltage,min_cell_voltage,max_cell,min_cell,avg_V
17787,MB7U8CLLTLJB30349,21-02-2022 09:41,31913,85.5,114,32.1,30.8,32.5,30.8,3.349,3.364,3.369,3.366,3.369,3.349,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.82,3.369,3.349,bv3,bv1,3.361000
17788,MB7U8CLLTLJB30349,21-02-2022 09:43,31913,86.8,114,32.2,30.9,32.6,31.0,3.353,3.366,3.369,3.369,3.370,3.356,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.58,3.370,3.353,bv7,bv1,3.363833
17789,MB7U8CLLTLJB30349,21-02-2022 09:45,31913,86.8,114,32.2,30.9,32.6,31.0,3.353,3.366,3.369,3.369,3.370,3.356,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.58,3.370,3.353,bv7,bv1,3.363833
17790,MB7U8CLLTLJB30349,21-02-2022 09:47,31913,88.3,114,32.3,31.1,32.7,31.1,3.354,3.367,3.370,3.370,3.374,3.364,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.45,3.374,3.354,bv7,bv1,3.366500
17791,MB7U8CLLTLJB30349,21-02-2022 09:49,31913,89.6,114,32.4,31.2,32.8,31.3,3.358,3.369,3.372,3.374,3.374,3.372,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.21,3.374,3.358,bv5,bv1,3.369833
17792,MB7U8CLLTLJB30349,21-02-2022 09:51,31913,89.6,114,32.4,31.2,32.8,31.3,3.358,3.369,3.372,3.374,3.374,3.372,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.21,3.374,3.358,bv5,bv1,3.369833
17793,MB7U8CLLTLJB30349,21-02-2022 09:53,31913,89.6,114,32.4,31.2,32.8,31.3,3.358,3.369,3.372,3.374,3.374,3.372,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.21,3.374,3.358,bv5,bv1,3.369833
17794,MB7U8CLLTLJB30349,21-02-2022 09:55,31913,91.8,114,32.6,31.4,33.0,31.5,3.359,3.372,3.375,3.375,3.377,3.382,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.07,3.382,3.359,bv8,bv1,3.373333
17795,MB7U8CLLTLJB30349,21-02-2022 09:57,31913,93.2,114,32.7,31.5,33.0,31.6,3.362,3.375,3.377,3.378,3.382,3.383,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.82,3.383,3.362,bv8,bv1,3.376167
17796,MB7U8CLLTLJB30349,21-02-2022 09:59,31913,93.2,114,32.7,31.5,33.0,31.6,3.362,3.375,3.377,3.378,3.382,3.383,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.82,3.383,3.362,bv8,bv1,3.376167


In [10]:
df[df['soc'] > 99]

,vin,intime,odometer,soc,cyclenum,bt1,bt2,bt3,bt4,bv1,bv2,bv3,bv5,bv7,bv8,bv17,bv18,bv19,bv20,bv21,bv22,bv23,bv24,ampherehour,max_cell_voltage,min_cell_voltage,max_cell,min_cell,avg_V
17800,MB7U8CLLTLJB30349,21-02-2022 10:09,31913,100.0,114,33.1,31.8,33.4,32.1,3.633,3.388,3.367,3.359,3.361,3.361,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.41,3.633,3.359,bv1,bv5,3.411500
17801,MB7U8CLLTLJB30349,21-02-2022 10:11,31913,100.0,114,33.1,31.8,33.4,32.1,3.633,3.388,3.367,3.359,3.361,3.361,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.41,3.633,3.359,bv1,bv5,3.411500
17802,MB7U8CLLTLJB30349,21-02-2022 10:13,31913,100.0,114,33.1,31.8,33.5,32.1,3.569,3.364,3.343,3.343,3.346,3.346,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.34,3.569,3.343,bv1,bv3,3.385167
17803,MB7U8CLLTLJB30349,21-02-2022 10:15,31913,100.0,114,33.2,31.8,33.5,32.1,3.536,3.356,3.338,3.340,3.343,3.345,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.01,3.536,3.338,bv1,bv3,3.376333
17804,MB7U8CLLTLJB30349,21-02-2022 10:21,31913,99.9,114,33.2,31.8,33.6,32.1,3.499,3.348,3.335,3.338,3.341,3.343,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.07,3.499,3.335,bv1,bv3,3.367333
17805,MB7U8CLLTLJB30349,21-02-2022 10:23,31913,99.9,114,33.2,31.8,33.6,32.1,3.478,3.345,3.333,3.337,3.340,3.341,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.10,3.478,3.333,bv1,bv3,3.362333


In [11]:
df = df.sort_values('intime').reset_index(drop = True)
df.loc[df['soc'] >= 99.9]['max_cell_voltage'].max()

3.633

In [12]:
x = df.loc[(df['soc'] == 100)].reset_index(drop = True)
highest_ah = x['ampherehour'][0]
highest_ah

23.41

In [13]:
if(x.empty):
    print("not enough data")
else:
    cell = x['max_cell'][0]
    print(cell)
    lowest_ah = df.loc[(round(df[cell],2) == 3.36)]['ampherehour'].min()
    X = highest_ah - lowest_ah
    rel_delta_capacity = (56.3 - X)/56.3
    print(X)
    print(rel_delta_capacity)

bv1
11.2
0.8010657193605684


In [14]:
Tsoh = (df[['bt1', 'bt2', 'bt3', 'bt4']].max()).max()
Tsoh

33.6

In [15]:
IC-((A**(-B/Tsoh)) * rel_delta_capacity)

99.5

## SOH Curve 

In [8]:
df = pd.read_csv("VIN_files/MB7U8CLLFLJL31151_output.csv")
df['year_month'] = (df['date']).astype(str)+"-"+((df['date.1']).astype(str)).str.zfill(2)
df

,date,date.1,ah_consumed,temp,odometer,Avg_Cell_temp_k,Relative_Partial_Ah,SOH Calc,SOH_km_Avg,year_month
0,2021,2,43.662000,33.320000,445.0,306.320000,0.113642,95.640243,95.640243,2021-02
1,2021,3,41.905926,38.044444,2756.0,311.044444,0.149291,93.795807,94.093620,2021-03
2,2021,4,41.987308,38.169231,5526.0,311.169231,0.147639,93.841633,93.967307,2021-04
3,2021,5,40.762500,35.079167,7484.0,308.079167,0.172503,93.375884,93.812576,2021-05
4,2021,6,40.240000,36.218519,10291.0,309.218519,0.183110,92.812042,93.539668,2021-06
5,2021,7,39.506389,33.936111,13354.0,306.936111,0.198003,92.669527,93.340084,2021-07
6,2021,8,39.746563,33.578125,16291.0,306.578125,0.193127,92.897638,93.260318,2021-08
7,2021,9,38.601212,34.581818,19510.0,307.581818,0.216378,91.913402,93.038088,2021-09
8,2021,10,36.931395,35.130233,23186.0,308.130233,0.250276,90.603464,92.652093,2021-10
9,2021,11,34.629444,34.952778,26609.0,307.952778,0.297007,88.989119,92.180885,2021-11


In [30]:
df.iloc[-1]['SOH_km_Avg']

87.0635208025963

In [28]:
import plotly_express as px
data = df.copy()
data['SOH_km_Avg'] = round(data['SOH_km_Avg'], 2)
fig = px.line(data, x="year_month", y="SOH_km_Avg", title='SOH', markers=True, text="SOH_km_Avg")
# Edit the layout
fig.update_layout(xaxis_title='Month',
                   yaxis_title='SOH')
fig.update_traces(textposition="top center", textfont_size=10)
fig.show()